# Chip TCO Comparison Agent

> **Llama 70B inference, in 30 seconds.**

This notebook walks through the canonical query end-to-end:

> *"Llama 3.1 70B inference, 100M tokens/day, p99 <500ms, US-East, 24-month amortization, no budget cap"*

The agent orchestrates 8–12 calls across the six **Silicon Analysts MCP tools** plus a bundled `cloud_prices.json` snapshot, and returns a ranked, provenance-tagged recommendation across cloud (AWS / Azure / GCP / CoreWeave / Lambda) and on-prem options. The whole loop is the **raw Anthropic SDK + the official MCP Python SDK** — no LangGraph, no Claude Agent SDK, no abstraction layers.

Every cell below is a small, readable piece of the same agent that ships in `chip_tco_agent.py`. The notebook *imports* from that module rather than re-implementing the loop, so what you see here is what the CLI runs.


## 1. Setup

Install the dependencies (skip if you already ran `uv sync`) and import what we need.

In [ ]:
# Idempotent — skip if you already ran `uv sync`.
%pip install -q anthropic mcp pydantic rich python-dotenv

import asyncio
import os
from rich.console import Console
from rich.markdown import Markdown

# All agent logic lives in chip_tco_agent.py — we import, not re-implement.
import chip_tco_agent as agent
from chip_tco_agent import (
    SYSTEM_PROMPT,
    TCORecommendation,
    build_tools_array,
    compute_tco,
    connect_mcp,
    enforce_confidence_min,
    lookup_cloud_price,
    mcp_tools_to_anthropic_tools,
    render_recommendation,
    run_agent,
    run_query,
)

console = Console()
console.print(f"[dim]chip_tco_agent loaded · max_turns={agent.DEFAULT_MAX_TURNS} · model={agent.DEFAULT_MODEL}[/dim]")


## 2. API Key Configuration

The agent needs two credentials:

- `ANTHROPIC_API_KEY` — for Claude (the LLM doing the reasoning). Get one at <https://console.anthropic.com>.
- `SILICON_ANALYSTS_API_KEY` — for the MCP server that serves chip-level data. Free tier ≈ **10 queries/day** (100 API calls/day, ~10 calls per query). Get one at <https://siliconanalysts.com/developers>.

Put them in `.env` (see `.env.example`) or your shell.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY"
assert os.environ.get("SILICON_ANALYSTS_API_KEY"), "Missing SILICON_ANALYSTS_API_KEY"

console.print("[green]✓[/green] credentials loaded")
console.print(f"[dim]MCP endpoint: {os.environ.get('SILICON_ANALYSTS_MCP_URL', agent.DEFAULT_MCP_URL)}[/dim]")


## 3. MCP Client Initialization

Connect to the Silicon Analysts MCP server using the **official `mcp` Python SDK** with Streamable HTTP transport and Bearer auth. List the six tools the server exposes — these are the authoritative chip-economics calls the agent will make.

In [ ]:
sa_key = os.environ["SILICON_ANALYSTS_API_KEY"]
mcp_url = os.environ.get("SILICON_ANALYSTS_MCP_URL", agent.DEFAULT_MCP_URL)

# Use the connect_mcp() context manager from chip_tco_agent. It wraps
# streamablehttp_client + ClientSession + initialize() + a 10s timeout.
async def show_mcp_tools():
    async with connect_mcp(sa_key, url=mcp_url) as session:
        result = await session.list_tools()
        for tool in result.tools:
            console.print(f"  [cyan]{tool.name}[/cyan] — {(tool.description or '').splitlines()[0][:90]}")

await show_mcp_tools()


## 4. Tool Adapter (MCP → Anthropic format)

This is the canonical educational moment of the notebook.

MCP and Anthropic both speak JSON Schema for tool definitions, but the field names differ:

| MCP | Anthropic |
|---|---|
| `inputSchema` | `input_schema` |

Everything else is identical. The adapter is ~15 lines:

In [ ]:
# inspect the implementation
import inspect
console.print(Markdown(f"```python\n{inspect.getsource(mcp_tools_to_anthropic_tools)}\n```"))


In [ ]:
# Discover MCP tools, adapt them, append the local tools, and inspect the combined array.
async def show_tools_array():
    async with connect_mcp(sa_key, url=mcp_url) as session:
        mcp_tools = (await session.list_tools()).tools
        all_tools = build_tools_array(mcp_tools)
        for t in all_tools:
            origin = "MCP" if t["name"] not in {"lookup_cloud_price", "compute_tco", "respond_with_recommendation"} else "local"
            console.print(f"  [{('cyan' if origin == 'MCP' else 'yellow')}]{t['name']}[/] [dim]({origin})[/dim]")
        return all_tools

tools_array = await show_tools_array()
console.print(f"\n[dim]{len(tools_array)} total tools available to the agent[/dim]")


## 5. Local Helpers

Two thin functions that the agent calls just like an MCP tool — they read the bundled JSON snapshots and do arithmetic.

- `lookup_cloud_price(chip, providers)` — reads `cloud_prices.json`. Returns `as_of`, per-cell pricing tiers, GA status, and a staleness warning if the snapshot is >60 days old.
- `compute_tco(deployment, chip, qty_gpus, ...)` — capex + opex + depreciation arithmetic. For on-prem, applies electricity / PUE / colo / OEM support / staff / software per `onprem_assumptions.json`.

In [ ]:
# Sample call: cloud pricing for H100 across the providers
result = lookup_cloud_price("H100", ["AWS", "Azure", "GCP", "Lambda", "CoreWeave"])

console.print(f"[bold]as_of:[/bold] {result['as_of']} ({result['snapshot_age_days']} days old)")
for provider, cell in result["data"].items():
    if cell.get("on_demand_per_gpu_hr"):
        console.print(f"  {provider:<10} OD ${cell['on_demand_per_gpu_hr']:>5.2f}  3yr ${cell.get('3yr_reserved_per_gpu_hr') or 0:>5.2f}/GPU-hr  ({cell.get('ga_status', '?')})")


In [ ]:
# Sample call: 24-month TCO for H100 × 4 on CoreWeave 3yr reserved (the spec's worked example)
tco = compute_tco(
    deployment="cloud",
    chip="H100",
    qty_gpus=4,
    horizon_months=24,
    price_per_gpu_hr=2.46,  # CoreWeave 3yr reserved per cloud_prices.json
    daily_tokens=100_000_000,
)
for k, v in tco.items():
    console.print(f"  [dim]{k}:[/dim] {v}")


## 6. System Prompt

The full system prompt is reproduced verbatim from spec Section E (with the turn cap adjusted to 10 — see `docs/design.md`). It's a module-level constant in `chip_tco_agent.py`, not a hidden f-string buried in agent code, so it's easy to read, audit, and edit.

The five hard rules are the trust contract: no invented numbers, confidence = `min()` of contributing tiers, hard turn cap, GA-status flagging, and explicit SLA interpretation.

In [ ]:
console.print(Markdown(f"""```\n{SYSTEM_PROMPT}\n```"""))


## 7. Agent Loop

`run_agent()` is the single-agent ReAct loop. Each turn:

1. Sends the conversation to Claude with all tools available.
2. Streams back text (the agent's thinking) and tool_use blocks.
3. Dispatches tool calls in parallel via `asyncio.gather`.
4. Appends tool_results and continues.
5. When the agent calls `respond_with_recommendation`, validates the payload against the Pydantic schema and returns.

Two safety rails the user gets for free:

- **Final-turn forcing**: at turn ≥ 8, `tool_choice` is set to `respond_with_recommendation` so the agent cannot drift past the cap.
- **Trust contract enforcement**: after the agent returns, `enforce_confidence_min()` clamps `confidence.overall` to the minimum of contributing tiers, even if the model claims higher. If we override, a caveat is appended.

In [ ]:
import inspect
console.print(Markdown(f"""```python\n{inspect.getsource(run_agent)}\n```"""))


## 8. The Headline Query

> *"Llama 3.1 70B inference, 100M tokens/day, p99 <500ms, US-East, 24-month amortization, no budget cap"*

Expected: rank #1 should be H100 or H200 on a neocloud (CoreWeave is cheapest in our snapshot), with B200 as a runner-up flagged for benchmark uncertainty. Total runtime: ~30–60s on Sonnet 4.5. Cost: ~$0.075 first run, ~$0.04 with caching.

In [ ]:
HEADLINE_QUERY = (
    "Llama 3.1 70B inference, 100M tokens/day, p99 <500ms, "
    "US-East, 24-month amortization, no budget cap"
)

result, metrics = await run_query(HEADLINE_QUERY, console=console)


## 9. Additional Example Queries

Three more workloads — each demonstrates a different shape of the agent's reasoning. These each use ~10 free-tier API calls; budget your daily quota accordingly.

In [ ]:
# 8B fine-tune within budget
await run_query(
    "Llama 3.1 8B fine-tune over 1B tokens, budget $50K, 30-day timeline",
    console=console,
)


In [ ]:
# 7B edge inference with strict TTFT
await run_query(
    "Llama 3.1 7B inference, 10ms TTFT target, EU region, low concurrency edge deployment",
    console=console,
)


In [ ]:
# Mixed workload (LLM + image gen)
await run_query(
    "Mixed workload: 50% Llama 70B inference, 50% Stable Diffusion XL image gen, 24-month horizon",
    console=console,
)


## 10. What to Try Next

Five prompts that exercise different corners of the agent:

1. *"Llama 3.1 405B inference at high concurrency — H100 vs H200 vs B200 vs MI300X."*
2. *"On-prem 8x H100 vs CoreWeave 3yr reserved at 90% utilization — what's the break-even?"*
3. *"GB200 NVL72 for a 1T-parameter MoE training run — is it worth the supply-chain risk?"*
4. *"What's the cheapest 7B inference under 50ms p99 in ap-northeast-1?"*
5. *"Compare Trainium2 vs H100 for a fine-tuning job that's portable across providers."*

To extend the agent:

- **Add your own region.** Edit `cloud_prices.json` to add an `eu-west-1` cell; the agent will pick it up on the next run.
- **Tighten the on-prem assumptions.** Edit `onprem_assumptions.json` — your colo $/kW/mo, your industrial electricity rate, your staff allocation.
- **Different model.** `export ANTHROPIC_MODEL=claude-opus-4-5` for harder queries (~5× cost, longer reasoning).
- **CLI mode.** `uv run python chip_tco_agent.py "<your query>"` runs the same loop with the same Rich rendering.

## 11. Footer

MIT licensed. Issue tracker: <https://github.com/silicon-analysts/chip-tco-agent/issues>.

The bundled `cloud_prices.json` is a snapshot dated `2026-04-30`. The agent warns when the snapshot is >60 days old. PRs that update prices for your region are welcome.

Built by [Silicon Analysts](https://siliconanalysts.com) — semiconductor data and analysis platform with an MCP server for AI agents.